## 🧩 CASE 2: Airline Customer Segmentation & Satisfaction Prediction

**Goal:**
Analyze airline passenger satisfaction data to identify what factors influence satisfaction and segment passengers into meaningful groups (loyal vs disloyal, business vs personal).[](url)

### Step 1: Load & Inspect Data

In [0]:
df = spark.table("workspace.default.`2_airline_passenger_satisfaction`")
display(df.limit(5))
df.printSchema()

id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
19556,Female,Loyal Customer,52,Business travel,Eco,160,5,4,3,4,3,4,3,5,5,5,5,2,5,5,50,44.0,satisfied
90035,Female,Loyal Customer,36,Business travel,Business,2863,1,1,3,1,5,4,5,4,4,4,4,3,4,5,0,0.0,satisfied
12360,Male,disloyal Customer,20,Business travel,Eco,192,2,0,2,4,2,2,2,2,4,1,3,2,2,2,0,0.0,neutral or dissatisfied
77959,Male,Loyal Customer,44,Business travel,Business,3377,0,0,0,2,3,4,4,1,1,1,1,3,1,4,0,6.0,satisfied
36875,Female,Loyal Customer,49,Business travel,Eco,1182,2,3,4,3,4,1,2,2,2,2,2,4,2,4,0,20.0,satisfied


root
 |-- id: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Customer Type: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Type of Travel: string (nullable = true)
 |-- Class: string (nullable = true)
 |-- Flight Distance: long (nullable = true)
 |-- Inflight wifi service: long (nullable = true)
 |-- Departure/Arrival time convenient: long (nullable = true)
 |-- Ease of Online booking: long (nullable = true)
 |-- Gate location: long (nullable = true)
 |-- Food and drink: long (nullable = true)
 |-- Online boarding: long (nullable = true)
 |-- Seat comfort: long (nullable = true)
 |-- Inflight entertainment: long (nullable = true)
 |-- On-board service: long (nullable = true)
 |-- Leg room service: long (nullable = true)
 |-- Baggage handling: long (nullable = true)
 |-- Checkin service: long (nullable = true)
 |-- Inflight service: long (nullable = true)
 |-- Cleanliness: long (nullable = true)
 |-- Departure Delay in Minutes: long (nullable = true)
 

### Step 2: Clean & Standardize Columns
Let’s fix inconsistent names and prepare numeric types.

In [0]:
from pyspark.sql.functions import col

# Rename columns for easier use
df = (df
      .withColumnRenamed("Customer Type", "Customer_Type")
      .withColumnRenamed("Type of Travel", "Type_of_Travel")
      .withColumnRenamed("Flight Distance", "Flight_Distance")
      .withColumnRenamed("Departure/Arrival time convenient", "Dep_Arr_Convenient")
      .withColumnRenamed("Ease of Online booking", "Ease_Online_Booking")
      .withColumnRenamed("Gate location", "Gate_Location")
      .withColumnRenamed("Food and drink", "Food_and_Drink")
      .withColumnRenamed("Online boarding", "Online_Boarding")
      .withColumnRenamed("Seat comfort", "Seat_Comfort")
      .withColumnRenamed("Inflight entertainment", "Inflight_Entertainment")
      .withColumnRenamed("On-board service", "Onboard_Service")
      .withColumnRenamed("Leg room service", "Legroom_Service")
      .withColumnRenamed("Baggage handling", "Baggage_Handling")
      .withColumnRenamed("Checkin service", "Checkin_Service")
      .withColumnRenamed("Inflight service", "Inflight_Service")
      .withColumnRenamed("Departure Delay in Minutes", "Dep_Delay")
      .withColumnRenamed("Arrival Delay in Minutes", "Arr_Delay")
)

df.printSchema()


root
 |-- id: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Type_of_Travel: string (nullable = true)
 |-- Class: string (nullable = true)
 |-- Flight_Distance: long (nullable = true)
 |-- Inflight wifi service: long (nullable = true)
 |-- Dep_Arr_Convenient: long (nullable = true)
 |-- Ease_Online_Booking: long (nullable = true)
 |-- Gate_Location: long (nullable = true)
 |-- Food_and_Drink: long (nullable = true)
 |-- Online_Boarding: long (nullable = true)
 |-- Seat_Comfort: long (nullable = true)
 |-- Inflight_Entertainment: long (nullable = true)
 |-- Onboard_Service: long (nullable = true)
 |-- Legroom_Service: long (nullable = true)
 |-- Baggage_Handling: long (nullable = true)
 |-- Checkin_Service: long (nullable = true)
 |-- Inflight_Service: long (nullable = true)
 |-- Cleanliness: long (nullable = true)
 |-- Dep_Delay: long (nullable = true)
 |-- Arr_Delay: double (nullable = tru

### Step 3: Handle Missing Values & Encode Labels
We’ll fill missing delay values and convert satisfaction into numeric.

In [0]:
from pyspark.sql.functions import when

# Fill missing numeric columns with 0
df = df.fillna(0)

# Convert satisfaction to binary label
df = df.withColumn(
    "Satisfaction_Label",
    when(col("satisfaction") == "satisfied", 1).otherwise(0)
)

display(df.select("satisfaction", "Satisfaction_Label").distinct())


satisfaction,Satisfaction_Label
satisfied,1
neutral or dissatisfied,0


### Step 4: SQL Exploration (for EDA & PPT Insights)

In [0]:
df.createOrReplaceTempView("passengers")

# Satisfaction by travel type
spark.sql("""
SELECT Type_of_Travel,
       ROUND(AVG(Satisfaction_Label)*100,2) AS satisfaction_rate,
       COUNT(*) AS total_passengers
FROM passengers
GROUP BY Type_of_Travel
ORDER BY satisfaction_rate DESC
""").show()

# Satisfaction by class
spark.sql("""
SELECT Class,
       ROUND(AVG(Satisfaction_Label)*100,2) AS satisfaction_rate
FROM passengers
GROUP BY Class
ORDER BY satisfaction_rate DESC
""").show()


+---------------+-----------------+----------------+
| Type_of_Travel|satisfaction_rate|total_passengers|
+---------------+-----------------+----------------+
|Business travel|            58.82|           18038|
|Personal Travel|             9.99|            7938|
+---------------+-----------------+----------------+

+--------+-----------------+
|   Class|satisfaction_rate|
+--------+-----------------+
|Business|            69.52|
|Eco Plus|            24.78|
|     Eco|            19.39|
+--------+-----------------+



### Step 5: Feature Preparation for Modeling
We’ll predict Satisfaction_Label using selected features.

**Categorical:** Gender, Customer_Type, Type_of_Travel, Class                        
**Numeric:** Flight_Distance, Dep_Delay, Arr_Delay, Seat_Comfort, Online_Boarding, Food_and_Drink, Inflight_Entertainment, Cleanliness

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

categorical_cols = ["Gender", "Customer_Type", "Type_of_Travel", "Class"]
indexers = [StringIndexer(inputCol=col, outputCol=col+"_idx", handleInvalid="keep") for col in categorical_cols]
encoders = [OneHotEncoder(inputCols=[col+"_idx"], outputCols=[col+"_ohe"]) for col in categorical_cols]

numeric_cols = ["Flight_Distance", "Dep_Delay", "Arr_Delay", "Seat_Comfort", "Online_Boarding", "Food_and_Drink", "Inflight_Entertainment", "Cleanliness"]
feature_cols = numeric_cols + [c+"_ohe" for c in categorical_cols]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")


### Step 6: Model Training (Logistic Regression)
We’ll use a classification model to predict satisfaction.

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="Satisfaction_Label")

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
pred = model.transform(test)


### Step 7: Evaluate Model Performance

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="Satisfaction_Label", metricName="areaUnderROC")
auc = evaluator.evaluate(pred)
print("✅ Model ROC AUC:", round(auc, 3))

# Simple accuracy check
correct = pred.filter(pred.prediction == pred.Satisfaction_Label).count()
total = pred.count()
print("✅ Accuracy:", round(correct/total, 3))


✅ Model ROC AUC: 0.902
✅ Accuracy: 0.839


### Step 8: Save Predictions Table (for SQL & Visualization)

In [0]:
pred.select("Gender","Customer_Type","Type_of_Travel","Class","Satisfaction_Label","prediction") \
    .write.mode("overwrite").saveAsTable("customer_satisfaction_predictions")

print("✅ Predictions saved as table: customer_satisfaction_predictions")


✅ Predictions saved as table: customer_satisfaction_predictions


In [0]:
%sql
SELECT Class, ROUND(AVG(prediction),2) AS Predicted_Satisfaction
FROM customer_satisfaction_predictions
GROUP BY Class
ORDER BY Predicted_Satisfaction DESC;


Class,Predicted_Satisfaction
Business,0.75
Eco Plus,0.19
Eco,0.11


Databricks visualization. Run in Databricks to view.

### Step 9: Clustering for Customer Segmentation (K-Means)
We’ll segment passengers into clusters based on service scores.

In [0]:
from pyspark.ml.clustering import KMeans

cluster_features = ["Seat_Comfort","Online_Boarding","Food_and_Drink","Inflight_Entertainment","Cleanliness","Flight_Distance"]
assembler2 = VectorAssembler(inputCols=cluster_features, outputCol="features_cluster")
df_cluster = assembler2.transform(df)

kmeans = KMeans(featuresCol="features_cluster", k=3, seed=1)
model_cluster = kmeans.fit(df_cluster)
df_clusters = model_cluster.transform(df_cluster)

display(df_clusters.select("Gender","Type_of_Travel","Class","satisfaction","prediction"))


Gender,Type_of_Travel,Class,satisfaction,prediction
Female,Business travel,Eco,satisfied,0
Female,Business travel,Business,satisfied,2
Male,Business travel,Eco,neutral or dissatisfied,0
Male,Business travel,Business,satisfied,2
Female,Business travel,Eco,satisfied,1
Male,Business travel,Eco,satisfied,0
Female,Business travel,Business,satisfied,2
Female,Business travel,Business,satisfied,2
Male,Business travel,Eco,satisfied,0
Female,Business travel,Business,satisfied,1
